In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import sys
import os

sys.path.append('..')
from src.data_processor import DataProcessor
from src.clustering import MusicClusterer

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print("✓ Libraries loaded")

## 1. Load and Prepare Data

In [ ]:
# Load data
processor = DataProcessor()

# Try to load processed data, otherwise create sample
try:
    df = pd.read_csv('../data/processed/cleaned_music_data.csv')
    print(f"Loaded {len(df)} tracks from processed data")
except:
    df = processor.create_sample_data(n_samples=5000)
    print(f"Created {len(df)} synthetic tracks")

df.head()

In [ ]:
# Select features for clustering
clustering_features = ['danceability', 'energy', 'loudness', 'acousticness', 'valence', 'tempo']
available_features = [f for f in clustering_features if f in df.columns]

print(f"Using features: {available_features}")

# Extract and scale features
X = df[available_features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nFeature matrix shape: {X_scaled.shape}")

## 2. Dimensionality Reduction (PCA)

In [ ]:
# PCA for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_)*100:.2f}%")

In [ ]:
# Visualize PCA components
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Explained variance
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)
cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)

axes[0].bar(range(1, len(pca_full.explained_variance_ratio_)+1), 
            pca_full.explained_variance_ratio_, alpha=0.7, label='Individual')
axes[0].plot(range(1, len(cumulative_var)+1), cumulative_var, 'ro-', label='Cumulative')
axes[0].axhline(y=0.95, color='g', linestyle='--', label='95% threshold')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('PCA Explained Variance')
axes[0].legend()

# 2D projection
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.5, s=10)
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].set_title('2D PCA Projection')

plt.tight_layout()
plt.show()

In [ ]:
# Feature loadings
loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=available_features
)
print("PCA Feature Loadings:")
print(loadings.round(3))

# Biplot
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.3, s=10)

for i, feature in enumerate(available_features):
    ax.arrow(0, 0, loadings.iloc[i, 0]*3, loadings.iloc[i, 1]*3, 
             head_width=0.1, head_length=0.1, fc='red', ec='red')
    ax.text(loadings.iloc[i, 0]*3.2, loadings.iloc[i, 1]*3.2, feature, fontsize=10)

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('PCA Biplot')
plt.show()

## 3. Finding Optimal Number of Clusters

In [ ]:
# Initialize clusterer
clusterer = MusicClusterer(random_state=42)

# Find optimal k
results = clusterer.find_optimal_k(X_scaled, k_range=(2, 10))

# Plot elbow curves
fig = clusterer.plot_elbow(results)
plt.savefig('../data/processed/elbow_analysis.png', dpi=150)
plt.show()

In [ ]:
# Summary table
results_df = pd.DataFrame(results)
results_df['silhouette_rank'] = results_df['silhouette'].rank(ascending=False)
results_df['db_rank'] = results_df['davies_bouldin'].rank(ascending=True)  # Lower is better
results_df['ch_rank'] = results_df['calinski_harabasz'].rank(ascending=False)
results_df['avg_rank'] = (results_df['silhouette_rank'] + results_df['db_rank'] + results_df['ch_rank']) / 3

print("Optimal K Analysis:")
print(results_df[['k', 'silhouette', 'davies_bouldin', 'calinski_harabasz', 'avg_rank']].round(3))
print(f"\n✓ Recommended k = {results_df.loc[results_df['avg_rank'].idxmin(), 'k']:.0f}")

## 4. K-Means Clustering

In [ ]:
# Fit K-Means with optimal k
optimal_k = 5  # Adjust based on elbow analysis
kmeans_labels = clusterer.fit_kmeans(X_scaled, n_clusters=optimal_k)

# Add labels to dataframe
df['kmeans_cluster'] = kmeans_labels

# Evaluate
kmeans_metrics = clusterer.evaluate_clustering(X_scaled, kmeans_labels)
print("\nK-Means Metrics:")
for metric, value in kmeans_metrics.items():
    if metric != 'cluster_sizes':
        print(f"  {metric}: {value:.4f}" if isinstance(value, float) else f"  {metric}: {value}")

In [ ]:
# Visualize K-Means clusters
fig = px.scatter(x=X_pca[:, 0], y=X_pca[:, 1], 
                 color=kmeans_labels.astype(str),
                 title='K-Means Clustering Results',
                 labels={'x': 'PC1', 'y': 'PC2', 'color': 'Cluster'},
                 opacity=0.6)
fig.show()

In [ ]:
# Cluster profiles
X_df = pd.DataFrame(X_scaled, columns=available_features)
kmeans_profiles = clusterer.get_cluster_profiles(X_df, kmeans_labels)

print("K-Means Cluster Profiles (standardized):")
print(kmeans_profiles.round(2))

# Original scale profiles
df_with_clusters = df.copy()
original_profiles = df_with_clusters.groupby('kmeans_cluster')[available_features].mean()
print("\nCluster Profiles (original scale):")
print(original_profiles.round(2))

In [ ]:
# Interpret clusters
interpretations = clusterer.interpret_clusters(original_profiles)
print("\nCluster Interpretations:")
for cluster_id, description in interpretations.items():
    size = kmeans_metrics['cluster_sizes'].get(cluster_id, 0)
    print(f"  Cluster {cluster_id}: {description} ({size} songs)")

In [ ]:
# Radar chart for cluster profiles
fig = go.Figure()

for cluster in original_profiles.index:
    # Normalize to 0-1 for visualization
    values = original_profiles.loc[cluster].copy()
    for col in values.index:
        if col == 'loudness':
            values[col] = (values[col] + 60) / 60  # Normalize loudness
        elif col == 'tempo':
            values[col] = values[col] / 200  # Normalize tempo
    
    fig.add_trace(go.Scatterpolar(
        r=values.values.tolist() + [values.values[0]],
        theta=values.index.tolist() + [values.index[0]],
        name=f'Cluster {cluster}: {interpretations.get(cluster, "")}'
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    showlegend=True,
    title='Cluster Profiles Radar Chart'
)
fig.show()

## 5. Hierarchical Clustering

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

# Create linkage matrix (on sample for visualization)
sample_idx = np.random.choice(len(X_scaled), min(500, len(X_scaled)), replace=False)
X_sample = X_scaled[sample_idx]

linkage_matrix = linkage(X_sample, method='ward')

# Dendrogram
plt.figure(figsize=(15, 8))
dendrogram(linkage_matrix, truncate_mode='level', p=5)
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Sample Index')
plt.ylabel('Distance')
plt.axhline(y=20, color='r', linestyle='--', label='Cut threshold')
plt.legend()
plt.show()

In [ ]:
# Fit hierarchical clustering
hierarchical_labels = clusterer.fit_hierarchical(X_scaled, n_clusters=optimal_k, linkage='ward')
df['hierarchical_cluster'] = hierarchical_labels

# Evaluate
hierarchical_metrics = clusterer.evaluate_clustering(X_scaled, hierarchical_labels)
print("Hierarchical Clustering Metrics:")
for metric, value in hierarchical_metrics.items():
    if metric != 'cluster_sizes':
        print(f"  {metric}: {value:.4f}" if isinstance(value, float) else f"  {metric}: {value}")

In [ ]:
# Visualize hierarchical clusters
fig = px.scatter(x=X_pca[:, 0], y=X_pca[:, 1], 
                 color=hierarchical_labels.astype(str),
                 title='Hierarchical Clustering Results',
                 labels={'x': 'PC1', 'y': 'PC2', 'color': 'Cluster'},
                 opacity=0.6)
fig.show()

## 6. DBSCAN Clustering

In [ ]:
# Find optimal epsilon using k-distance graph
from sklearn.neighbors import NearestNeighbors

k = 5
nn = NearestNeighbors(n_neighbors=k)
nn.fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)
distances = np.sort(distances[:, k-1])

plt.figure(figsize=(10, 5))
plt.plot(distances)
plt.xlabel('Points')
plt.ylabel(f'{k}-th Nearest Neighbor Distance')
plt.title('K-Distance Graph for DBSCAN eps Selection')
plt.grid(True)
plt.show()

# Suggested eps (elbow point)
suggested_eps = np.percentile(distances, 90)
print(f"Suggested eps: {suggested_eps:.2f}")

In [ ]:
# Fit DBSCAN
dbscan_labels = clusterer.fit_dbscan(X_scaled, eps=suggested_eps, min_samples=5)
df['dbscan_cluster'] = dbscan_labels

# Evaluate (only if we have valid clusters)
n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
if n_clusters >= 2:
    dbscan_metrics = clusterer.evaluate_clustering(X_scaled, dbscan_labels)
    print("DBSCAN Metrics:")
    for metric, value in dbscan_metrics.items():
        if metric != 'cluster_sizes':
            print(f"  {metric}: {value:.4f}" if isinstance(value, float) else f"  {metric}: {value}")
else:
    print("DBSCAN found insufficient clusters. Try adjusting eps.")

In [ ]:
# Visualize DBSCAN
fig = px.scatter(x=X_pca[:, 0], y=X_pca[:, 1], 
                 color=dbscan_labels.astype(str),
                 title='DBSCAN Clustering Results (Noise = -1)',
                 labels={'x': 'PC1', 'y': 'PC2', 'color': 'Cluster'},
                 opacity=0.6)
fig.show()

## 7. Algorithm Comparison

In [ ]:
# Compare all algorithms
comparison = pd.DataFrame({
    'Algorithm': ['K-Means', 'Hierarchical', 'DBSCAN'],
    'Silhouette': [
        kmeans_metrics['silhouette_score'],
        hierarchical_metrics['silhouette_score'],
        dbscan_metrics.get('silhouette_score', np.nan) if n_clusters >= 2 else np.nan
    ],
    'Calinski-Harabasz': [
        kmeans_metrics['calinski_harabasz_score'],
        hierarchical_metrics['calinski_harabasz_score'],
        dbscan_metrics.get('calinski_harabasz_score', np.nan) if n_clusters >= 2 else np.nan
    ],
    'Davies-Bouldin': [
        kmeans_metrics['davies_bouldin_score'],
        hierarchical_metrics['davies_bouldin_score'],
        dbscan_metrics.get('davies_bouldin_score', np.nan) if n_clusters >= 2 else np.nan
    ],
    'N_Clusters': [
        kmeans_metrics['n_clusters'],
        hierarchical_metrics['n_clusters'],
        n_clusters
    ]
})

print("Algorithm Comparison:")
print(comparison.round(4))

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

titles = ['K-Means', 'Hierarchical', 'DBSCAN']
labels_list = [kmeans_labels, hierarchical_labels, dbscan_labels]

for ax, title, labels in zip(axes, titles, labels_list):
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis', alpha=0.5, s=10)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title(title)
    plt.colorbar(scatter, ax=ax)

plt.tight_layout()
plt.savefig('../data/processed/clustering_comparison.png', dpi=150)
plt.show()

## 8. Save Best Model

In [ ]:
# Save K-Means model (typically performs best)
clusterer.feature_names = available_features
model_path = '../models/clustering/kmeans_model.pkl'
clusterer.save_model(model_path, 'kmeans')

# Save clustered data
df.to_csv('../data/processed/music_data_clustered.csv', index=False)
print(f"\n✓ Saved clustered data with {len(df)} tracks")

## Summary

### Key Findings:
1. **Optimal Clusters**: Based on elbow analysis, k=5 provides good separation
2. **Best Algorithm**: K-Means typically provides the most balanced results
3. **Cluster Interpretations**: Clusters correspond to different mood/energy combinations

### Cluster Mapping to Emotions:
- High Energy + High Valence → Happy/Excited
- Low Energy + Low Valence → Sad/Melancholic
- High Energy + Low Valence → Angry/Intense
- Low Energy + High Valence → Calm/Peaceful
- Medium values → Neutral/Ambient